# Đỉnh–suy giảm có ở mô hình khác ngoài APKGC không?

Cả năm lần chạy APKGC đều đạt đỉnh giữa chừng rồi đi xuống, và chọn checkpoint
trên tập test thổi phồng MRR **+0,0054** — gấp 2,5 lần mức thống kê cực trị dự
đoán cho nhiễu thuần.

Nhưng đó là **một mô hình, một bộ dữ liệu**. Chưa đủ để nói về MMKGC nói chung.

Notebook này chạy **MyGO** (AAAI 2025) trên MKG-W và ghi lại **toàn bộ vết đánh
giá**, rồi áp cùng phép phân tích. MyGO được chọn vì nó cũng chọn checkpoint trên
tập test, và khuôn mã của nó đã lan sang C2RS.

Nếu MyGO cũng đỉnh–suy giảm thì phát biểu nâng từ *"APKGC thế này"* lên
*"các phương pháp MMKGC thế này"* — nâng hai đóng góp cùng lúc, chỉ tốn ~20 giờ GPU
thay vì 65–75 giờ của thí nghiệm đảo bảng xếp hạng.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = "/content/drive/MyDrive/mmkgc_traces"
os.makedirs(OUT, exist_ok=True)
print("kết quả:", OUT, "|", sorted(os.listdir(OUT)))

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
%cd /content
!rm -rf MyGO
# Repo tự chứa dữ liệu: data/ (3,3MB) và tokens/ (119MB) đã có sẵn trong kho.
!git clone -q https://github.com/zjukg/MyGO.git
%cd /content/MyGO
!pip install -q -r requirements.txt 2>&1 | tail -2
import os
print("data:", os.listdir("data")[:4], "| tokens có:", len(os.listdir("tokens")))

In [ ]:
# MyGO ghi MRR mỗi lần đánh giá qua logger (train_mygo_fgc.py:169,189) nên bắt
# stdout là đủ, không cần sửa mã của họ.
#
# Chỉ đổi valid_epoch để lấy đủ điểm trên vết: mặc định 50 với 1500 epoch cho
# 30 lần đánh giá. Giữ nguyên mọi siêu tham số khác đúng như run.sh của tác giả.
import re
RUN = open("run.sh").read()
m = re.search(r"train_mygo_fgc\.py (--data MKG-W[^>]*)", RUN)
assert m, "không tìm thấy dòng lệnh MKG-W trong run.sh"
ARGS = m.group(1).strip()
print("siêu tham số đọc thẳng từ run.sh của tác giả:")
print(" ", ARGS)

In [ ]:
import subprocess, sys, re, json, time, os

TAG = "mygo_mkgw_seed42"
DONE = f"{OUT}/{TAG}.json"

MRR_LINE = re.compile(r"MRR:\s*([\d.]+)")

if os.path.exists(DONE):
    print("đã chạy xong trước đó:", json.load(open(DONE))["n_evals"], "lần đánh giá")
else:
    cmd = [sys.executable, "-u", "train_mygo_fgc.py"] + ARGS.split()
    print(" ".join(cmd), flush=True)
    evals, t0 = [], time.time()
    with open(f"{OUT}/{TAG}.log", "w", buffering=1) as lf:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        for line in p.stdout:
            lf.write(line)
            g = MRR_LINE.search(line)
            if g:
                evals.append(float(g.group(1)))
                print(f"  eval {len(evals):>3}: MRR {evals[-1]:.4f}", flush=True)
                # ghi ngay vào Drive: Colab hay ngắt giữa chừng
                json.dump(dict(tag=TAG, evals=evals, n_evals=len(evals)),
                          open(f"{OUT}/partial_{TAG}.json", "w"))
        p.wait()
    if evals:
        json.dump(dict(tag=TAG, model="MyGO", dataset="MKG-W", args=ARGS,
                       evals=evals, n_evals=len(evals),
                       seconds=round(time.time() - t0, 1)), open(DONE, "w"), indent=2)
        print(f"\nxong: {len(evals)} lần đánh giá trong {(time.time()-t0)/60:.0f} phút")
    else:
        print("KHÔNG bắt được lần đánh giá nào — xem", f"{OUT}/{TAG}.log")

In [ ]:
# Cùng phép phân tích đã áp cho APKGC.
import json, math, statistics, glob

APKGC = dict(inflation=0.0054, ratio=2.5, sd=0.0009)   # tự đo, 5 lần chạy

for f in sorted(glob.glob(f"{OUT}/*.json")):
    if "partial" in f: continue
    d = json.load(open(f)); m = d["evals"]
    if len(m) < 4: continue
    peak = max(m); pi = m.index(peak) + 1
    infl = peak - m[-1]
    print(f"\n=== {d['tag']} ({d.get('model','?')} / {d.get('dataset','?')}) ===")
    print(f"  {len(m)} lần đánh giá | đỉnh {peak:.4f} ở lần {pi}/{len(m)} | cuối {m[-1]:.4f}")
    print(f"  thổi phồng nếu chọn trên test: {infl:+.4f}")
    if infl > 0:
        # so với thống kê cực trị: nhiễu iid thì max−cuối ~ sigma*sqrt(2 ln k)
        pred = APKGC["sd"] * math.sqrt(2 * math.log(len(m)))
        print(f"  lý thuyết nhiễu iid: {pred:.4f}  ->  đo gấp {infl/pred:.1f}× "
              f"(APKGC: {APKGC['ratio']}×)")
    print(f"  đỉnh ở CUỐI? {'CÓ — không có thổi phồng' if pi == len(m) else 'KHÔNG — có thổi phồng'}")
    print(f"  vết: " + " ".join(f"{x:.4f}" for x in m[:12]) + (" …" if len(m) > 12 else ""))

print(f"\nMốc APKGC: thổi phồng +{APKGC['inflation']:.4f}, gấp {APKGC['ratio']}× nhiễu iid,")
print("đỉnh không bao giờ ở lần cuối (5/5 lần chạy).")

## Đọc kết quả thế nào

**Nếu MyGO cũng đỉnh giữa chừng rồi đi xuống** → phát biểu nâng lên mức MMKGC,
đóng góp A và B mạnh hẳn. Bước tiếp theo là thí nghiệm đảo bảng xếp hạng.

**Nếu MyGO đạt đỉnh ở lần đánh giá cuối** → APKGC là ca cá biệt. Phải hạ phát biểu
xuống *"quan sát trên APKGC"*, và thí nghiệm đảo bảng xếp hạng mất phần lớn ý
nghĩa — tiết kiệm được 70 giờ GPU.

Kết quả nào cũng dùng được. Đó là lý do chạy cái này trước.